# 2 · Compare and decide

**What would make the repair acceptable?** A score is evidence; acceptance is a separately declared policy. We keep execution status, evaluator controls, review findings and the policy result distinct.

The repair changes only the retrieved policy. A fresh kernel can run this notebook independently. When notebook 01 already recorded the deployed batch in the same output directory, it is reused exactly once.


In [ ]:
import os, sys, json
from pathlib import Path

# Works in local Jupyter and a Databricks Git folder; notebooks have no __file__.
start = Path(os.environ.get("WORKSHOP_PROJECT_ROOT", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "west_workshop" / "data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set WORKSHOP_PROJECT_ROOT to the workshop checkout, then restart this notebook.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
MODE = os.environ.get("WORKSHOP_MODE", "saved")  # Explicit opt-in before any model calls.


In [ ]:
from IPython.display import display, HTML
from west_workshop.repair_workshop import notebook_session, paired_table, show_decision, render_report

session = notebook_session(MODE)
print(json.dumps(session.banner(), indent=2))


## Test the judge before interpreting a comparison

The original eight authored controls are preserved. Eight development controls encode known errors. Four separately authored controls were held back until the new rubric was frozen. Both positive and negative answers have explicit labels. Agreement on these 20 examples is a **smoke test**, not a representative accuracy estimate.

The old judge and its dated errors remain historical evidence. Failed controls are retained. A control failure prevents a readiness claim; we do not keep retrying for green. A recorded execution failure stops further provider work and stays visible in the decision below.


In [ ]:
controls = None
if session.has_execution_failure():
    print("Controls skipped: a recorded execution failed. Its evidence is preserved.")
else:
    controls = session.qualify_judge()
    print("Controls:", "PASSED" if controls["passed"] else "FAILED")
    print("Disagreements:", controls["disagreements"])
    print("Control version:", controls["version"])
    print("Smoke coverage only; no population accuracy claim.")


## Generate and evaluate the proposed repair

This is the same native MLflow API and the same cases and checks as notebook 01. Saved mode uses the frozen evidence. Live mode reuses any completed deployed batch and generates only missing batches. If the controls fail, it preserves that outcome and withholds the comparison.


In [ ]:
if MODE == "live" and controls and controls["passed"] and not session.has_execution_failure():
    import mlflow
    for variant in ("deployed", "repair"):
        if session.needs(variant):
            with session.evaluation(variant) as run:
                result = mlflow.genai.evaluate(
                    data=run.data, predict_fn=run.predict_fn, scorers=run.scorers,
                )
                session.record_evaluation(variant, result, run)
            if session.has_execution_failure():
                print("Further evaluation skipped: this batch has an execution failure.")
                break


## Code view 3: declare the policy, then apply it

Both answer sets must have complete comparable cases, valid binary scores and identical evaluator definitions. The candidate must pass every mandatory check and its absolute floor. Ordinary losses have a fixed allowance; protected-case losses cannot be offset by gains. The original assistant is allowed to contain the defect being repaired.

Controls must pass. A material unresolved evaluator finding requires review. A model review is labelled as such and is not a claim that a human signed off.


In [ ]:
from west_workshop.repair_policy import DEFAULT_POLICY, evaluate_repair

display({key: DEFAULT_POLICY[key] for key in (
    "version", "mandatory_checks", "quality_floor", "max_regression_rate", "protected_checks",
)})
review_path = os.environ.get("WORKSHOP_REVIEW_JSON")
reviewed = json.loads(Path(review_path).read_text(encoding="utf-8")) if review_path else None
comparison = session.policy_inputs(reviewed)
policy_result = evaluate_repair(**comparison)
report = session.finish(comparison["review"])
assert report["policy_result"] == policy_result


## Inspect the pairs

A paired row compares the **same case ID** before and after. Gains do not cancel protected failures. If evaluator evidence is invalid, no paired comparison is interpreted.


In [ ]:
display(HTML(paired_table(report)))


## Vote before revealing the decision

**Pause for 20 seconds:** PASS, BLOCK, REVIEW REQUIRED, or INVALID EVIDENCE? Name the evidence that would change your decision.

Keep the next output below the viewport until the vote. In a preexecuted saved notebook, collapse that output before presenting. The result is the actual recorded outcome, not a target attendees must reproduce.


In [ ]:
display(HTML(render_report(report)))
show_decision(report)


## Turn a finding into a future test

Preserve the original answer and score. Record the exact claim, expected behavior, reviewer and rationale. A human approves a new case label before promotion; automatic feedback is not ground truth. Add a versioned case, qualify a changed judge on controls, then rescore **all frozen compared answers** with the same new definition. Never patch one disagreeable score.

The take-home helpers are `review_skeleton`, `promote_reviewed_case` and `rescore_frozen` in `west_workshop/repair_workshop.py`. Rescoring uses `predict_fn=None`: it calls the new judge on fixed answers and records differences without generating new answers. The historical v1 decision stays unchanged.


## Carry this into your application

Replace `make_predictor` with your traced application adapter. Add reviewed cases with stable IDs. Keep reliable deterministic checks; qualify a judge for the meanings that need one. Declare your mandatory checks, floors, protected cases and review ownership before looking at a candidate.

```bash
python -m west_workshop evaluate --mode saved --evidence evidence/workshop/local/final
python -m west_workshop gate --mode saved --evidence evidence/workshop/local/final
```

`evaluate` exits **0** for a completed policy result, including BLOCK; **1** means execution failed. `gate` writes its report before enforcing: **0** PASS, **2** BLOCK, **3** REVIEW_REQUIRED, **4** INVALID, **1** execution failure / NOT_EVALUATED. The older `eval_gate.py` checks one named scorer and is a narrower contract.

**What remains outside this workshop:** representative held-out coverage, human validation of the judge, application security, operational limits, latency/cost budgets and actual production approval. Name who owns the next decision and what evidence they need.
